# Bài 1: AI là gì, không là gì
**Mở hộp đen AI · Tập 1 · Phòng thí nghiệm Colab**

Bốn chặng: **Chạy → Đọc → Sửa → Tự làm**. Bấm nút Chạy ở từng ô theo thứ tự từ trên xuống.
Phát hành tại: https://montinhoc.vn

## Chặng 1: Chạy
### Ô 1. Luật chơi oẳn tù tì

In [ ]:
import random
from collections import Counter

NUOC = ["búa", "kéo", "bao"]
THANG = {"búa": "kéo", "kéo": "bao", "bao": "búa"}   # búa thắng kéo, ...
KHAC = {"kéo": "búa", "bao": "kéo", "búa": "bao"}    # muốn thắng kéo thì ra búa, ...

def ket_qua(may, nguoi):
    # +1: máy thắng, 0: hoà, -1: máy thua
    if may == nguoi:
        return 0
    return 1 if THANG[may] == nguoi else -1

print(ket_qua("bao", "búa"), ket_qua("kéo", "kéo"), ket_qua("kéo", "búa"))

### Ô 2. Một người chơi có thói quen
Người thật hiếm khi ra đều tay. Ta giả lập một bạn hay ra búa: 50% búa, 30% kéo, 20% bao.

In [ ]:
def nguoi_choi():
    return random.choices(NUOC, weights=[50, 30, 20])[0]

random.seed(1)
thu = [nguoi_choi() for _ in range(10)]
print(thu)
print(Counter(thu))

### Ô 3. Ba con bot
- **Bot ngẫu nhiên**: ra bừa.
- **Bot theo luật**: người viết sẵn một luật cố định cho nó.
- **Bot biết học**: không ai viết luật, nó tự đếm thói quen của đối thủ.

In [ ]:
def bot_ngau_nhien(lich_su):
    return random.choice(NUOC)

def bot_theo_luat(lich_su):
    # LUẬT do người viết: "ra nước thắng được nước vừa rồi của đối thủ"
    if not lich_su:
        return "bao"
    return KHAC[lich_su[-1]]

def bot_biet_hoc(lich_su):
    # KHÔNG có luật viết sẵn: đếm xem đối thủ hay ra gì nhất
    if not lich_su:
        return random.choice(NUOC)
    hay_ra_nhat = Counter(lich_su).most_common(1)[0][0]
    return KHAC[hay_ra_nhat]

### Ô 4. Cho đấu 1.000 ván

In [ ]:
def dau(bot, so_van=1000, hat_giong=0):
    random.seed(hat_giong)
    lich_su, dem = [], Counter()
    for _ in range(so_van):
        may = bot(lich_su)          # bot chỉ được xem các ván ĐÃ QUA
        nguoi = nguoi_choi()
        dem[ket_qua(may, nguoi)] += 1
        lich_su.append(nguoi)
    return dem

cac_bot = [("ngẫu nhiên", bot_ngau_nhien), ("theo luật", bot_theo_luat),
           ("biết học", bot_biet_hoc)]
for ten, bot in cac_bot:
    d = dau(bot)
    print(f"Bot {ten:10} | thắng {d[1]/10:.1f}% | hoà {d[0]/10:.1f}%"
          f" | thua {d[-1]/10:.1f}%")

## Chặng 2: Đọc
Trả lời vào vở trước khi đi tiếp:
1. Trong `bot_theo_luat`, dòng nào là luật do người viết? Trong `bot_biet_hoc`, 'kiến thức' của bot nằm ở đâu?
2. Vì sao bot ngẫu nhiên thắng xấp xỉ một phần ba số ván?
3. Bot biết học có 'hiểu' trò oẳn tù tì không? Em dựa vào đâu để nói vậy?

## Chặng 3: Sửa
### Ô 5. Số ván ít thì con số nhảy lung tung

In [ ]:
for so_van in [10, 100, 1000, 10000]:
    d = dau(bot_biet_hoc, so_van)
    print(f"{so_van:>6} ván: bot biết học thắng {d[1]/so_van:.1%}")

### Ô 6. Đổi thói quen của người chơi
Sửa `weights` thành `[20, 30, 50]` (hay ra bao) rồi chạy lại. Bot nào tự thích nghi, bot nào không cần thích nghi?

In [ ]:
def nguoi_choi():
    return random.choices(NUOC, weights=[20, 30, 50])[0]

for ten, bot in [("theo luật", bot_theo_luat), ("biết học", bot_biet_hoc)]:
    d = dau(bot)
    print(f"Bot {ten:10} | thắng {d[1]/10:.1f}% | thua {d[-1]/10:.1f}%")

### Ô 7. Một đối thủ biết tỏng con bot
Người chơi này biết bot đang đếm thói quen, nên cố tình ra đều rồi lật kèo.

In [ ]:
def dau_voi_nguoi_khon(so_van=1000):
    lich_su, dem = [], Counter()
    for _ in range(so_van):
        may = bot_biet_hoc(lich_su)
        # người khôn tự tính trước bot sẽ ra gì, rồi ra nước thắng nó
        nguoi = KHAC[may]
        dem[ket_qua(may, nguoi)] += 1
        lich_su.append(nguoi)
    return dem

random.seed(0)
d = dau_voi_nguoi_khon()
print(f"Bot biết học | thắng {d[1]/10:.1f}% | thua {d[-1]/10:.1f}%")

## Chặng 4: Tự làm
1. Chạy ô dưới để tự đấu 15 ván với bot biết học. Em có thắng được nó không? Em đã dùng mẹo gì?
2. Viết `bot_cua_em(lich_su)` theo ý tưởng riêng (ví dụ: chỉ đếm 5 ván gần nhất) và cho nó đấu bằng hàm `dau`.

In [ ]:
lich_su, diem = [], 0
for van in range(15):
    may = bot_biet_hoc(lich_su)
    nguoi = input(f"Ván {van + 1} - em ra gì (búa/kéo/bao)? ").strip().lower()
    if nguoi not in NUOC:
        print("Gõ lại nhé: búa, kéo hoặc bao"); continue
    kq = ket_qua(may, nguoi)
    diem -= kq
    print(f"  Máy ra {may}.", {1: "Máy thắng", 0: "Hoà", -1: "Em thắng"}[kq], "| Điểm của em:", diem)
    lich_su.append(nguoi)

**Nhật ký dùng AI:** Em có nhờ AI ở bước nào không? Nó sai ở đâu? Em sửa thế nào?

*(ghi vào đây)*